# ARC Prize 2026 -- C++ Graph Solver submission

**Code competition** -- this notebook runs **offline** (no internet) and must
write `/kaggle/working/submission.json`. Kaggle kills the kernel at **12h**.

The solver ships as **C++ source** in `code.tar.gz` (dataset
`lucafavole/arc-graph-solver-code`); this notebook extracts it, **compiles** it
with the image's `g++`, and runs the `arc_solver` binary. Shipping source (not a
prebuilt binary) compiles against Kaggle's exact toolchain, so there is no
glibc/architecture mismatch.

Runtime is bounded by a **per-task wall cap** (`--max-task-seconds`): the cap is
sized from the global budget so the run finishes and writes a *complete*
submission on its own. Attach two inputs before *Save & Run All*:
1. Competition data: **ARC Prize 2026 - ARC-AGI-2**
2. Solver code dataset: **lucafavole/arc-graph-solver-code**


In [ ]:
# === Setup: locate competition data + stage the C++ source ===================
import os, sys, glob, json, subprocess, time, tarfile, shutil, multiprocessing

WORK_DIR = "/kaggle/working"
SPLIT    = "test"   # "test" = real submission; "evaluation" = dry-run (has solutions)

def _find(pattern, root="/kaggle/input"):
    return sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))

# Competition data: locate the challenges file by name (folder name != slug).
ch_hits = _find(f"arc-agi_{SPLIT}_challenges.json")
assert ch_hits, (f"arc-agi_{SPLIT}_challenges.json not found -> attach the "
                 "'ARC Prize 2026 - ARC-AGI-2' competition data.")
challenges_file = ch_hits[0]
COMP_DIR = os.path.dirname(challenges_file)

# Solver source: prefer a shipped code.tar.gz; otherwise fall back to the raw
# csolver/ tree the code dataset ships. Either way stage a WRITABLE copy under
# /kaggle/working (/kaggle/input is read-only and `make` writes objects/binary).
extract_dir = os.path.join(WORK_DIR, "_code")
if os.path.isdir(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

arch_hits = _find("code.tar.gz")
if arch_hits:
    with tarfile.open(arch_hits[0]) as t:
        t.extractall(extract_dir)
    src_mode = "code.tar.gz"
else:
    mk_hits = _find("csolver/Makefile") or [p for p in _find("Makefile") if os.path.isdir(os.path.join(os.path.dirname(p), "src"))]
    if not mk_hits:
        print("=== /kaggle/input tree ===")
        for r, _, fs in os.walk("/kaggle/input"):
            print(r)
            for f in fs:
                print("   ", f)
        raise AssertionError("no code.tar.gz and no csolver/Makefile in inputs -> "
                             "attach the solver code dataset.")
    shutil.copytree(os.path.dirname(mk_hits[0]), os.path.join(extract_dir, "csolver"))
    src_mode = "raw tree"

mk = _find("Makefile", root=extract_dir)
assert mk, "Makefile not found after staging the solver source."
CSOLVER = os.path.dirname(mk[0])

print("Setup OK")
print("  COMP_DIR        =", COMP_DIR)
print("  challenges_file =", challenges_file)
print("  CSOLVER         =", CSOLVER)
print("  source mode     =", src_mode)


In [ ]:
# === Compile the C++ solver (offline, against Kaggle's own toolchain) =========
def _probe_compiler():
    for cxx in ("g++", "clang++"):
        try:
            subprocess.run([cxx, "--version"], check=True,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return cxx
        except Exception:
            continue
    return None

CXX = _probe_compiler()
assert CXX, ("no C++ compiler (g++/clang++) in the Kaggle image -> ship a prebuilt "
             "static x86_64 arc_solver binary in the dataset instead.")
print("compiler:", CXX)

# Portability shim: Kaggle's g++/libstdc++ does NOT transitively expose several
# standard symbols (size_t, uint*_t, memcpy, ...) that Apple clang/libc++ leaks,
# so the source compiles locally but fails on Kaggle. Force-include the standard
# headers into every translation unit by appending to COMMON_FLAGS -- this leaves
# the parity-critical flags (-ffp-contract=off etc.) intact.
import re as _re
_mkp = os.path.join(CSOLVER, "Makefile")
_src = open(_mkp).read()
_force = (" -include cstddef -include cstdint -include cstring -include cstdlib"
          " -include cstdio -include climits -include limits -include algorithm"
          " -include utility -include functional -include stdexcept")
_src2, _n = _re.subn(r"(?m)^(COMMON_FLAGS\s*:=.*)$", r"\1" + _force, _src, count=1)
assert _n == 1, "could not patch COMMON_FLAGS in Makefile"
open(_mkp, "w").write(_src2)
print("patched COMMON_FLAGS with force-includes")

t0 = time.time()
r = subprocess.run(["make", "-C", CSOLVER, "arc_solver", f"CXX={CXX}", "-j2"],
                   capture_output=True, text=True, errors="replace")
print(r.stdout[-2000:])
print(r.stderr[-2000:])
assert r.returncode == 0, "compile failed (see output above)"
ARC_SOLVER = os.path.join(CSOLVER, "arc_solver")
assert os.path.isfile(ARC_SOLVER), "arc_solver binary missing after build"

# v59: second binary from the v52 code (csolver_v52/, main 938ac13) for the anchor members
_v52 = [p for p in _find("csolver_v52/Makefile")]
assert _v52, "csolver_v52/ not found in the code dataset"
CSOLVER_V52 = os.path.join(WORK_DIR, "_code52", "csolver_v52")
if os.path.isdir(CSOLVER_V52):
    shutil.rmtree(CSOLVER_V52)
shutil.copytree(os.path.dirname(_v52[0]), CSOLVER_V52)
_mkp = os.path.join(CSOLVER_V52, "Makefile")
_s52, _n = _re.subn(r"(?m)^(COMMON_FLAGS\s*:=.*)$", r"\1" + _force, open(_mkp).read(), count=1)
assert _n == 1, "could not patch COMMON_FLAGS in the v52 Makefile"
open(_mkp, "w").write(_s52)
r = subprocess.run(["make", "-C", CSOLVER_V52, "arc_solver", f"CXX={CXX}", "-j2"],
                   capture_output=True, text=True, errors="replace")
print(r.stderr[-2000:])
assert r.returncode == 0, "v52 compile failed (see output above)"
ARC_SOLVER_V52 = os.path.join(CSOLVER_V52, "arc_solver")
assert os.path.isfile(ARC_SOLVER_V52), "v52 arc_solver binary missing after build"
print("Built", ARC_SOLVER_V52, "from", open(os.path.join(CSOLVER_V52, "SOURCE_COMMIT")).read().strip())
print(f"Built {ARC_SOLVER} in {time.time() - t0:.0f}s")


In [ ]:
# === floor-anchored ensemble (v51: v43 core + pop-160 members, dev-run skip) ===========================
# Config is EXHAUSTED: every single-config C++ (seed/pop/restarts/gens/max_rules)
# = 0.83; the multi-config ENSEMBLE (union into pass@2) = 1.67; widening it (3->5
# seeds, +max_rules) added nothing; SELECTION is verified futile on ARC-2 (its
# solves are gen-0 Phase-A SEEDS, and tournament/lexicase/lexicase_inject solve an
# IDENTICAL set). The ONE net-new lever is the mined-MACRO Phase-A seed lane
# (multi-rule composition -- exactly what ARC-2 needs and single-mechanism seeds
# lack), now wired into the deploy binary (main.cpp make_search_config loads a
# library when ARC_MACRO_SEED_LIBRARY is set; default OFF = byte-identical).
# DESIGN (floor cannot drop below the proven 1.67):
#   * members 0-2 = the PROVEN v43 3-seed core, macro OFF. member 0 (seed 777) is
#     the attempt_1 anchor -> 1.67 floor; 2718/333 carry the 2nd solve by consensus.
#   * members 3-4 = MACRO seed members sharing ONE library (macros_full, top-16).
#     Macro seeds are deterministic, so both macro members solve the same macro-
#     solvable tasks IDENTICALLY -> 2 consensus votes -> a generalizing macro solve
#     lands in attempt_2 (per-task) without displacing a proven solve.
import multiprocessing
N_WORKERS            = max(1, multiprocessing.cpu_count())
TOTAL_BUDGET_SECONDS = int(10.5 * 3600)

# Locate the macro library shipped inside the extracted code tree (csolver/library/).
_libs = _find("macros_full.json", root=CSOLVER) or _find("macros_full.json")
MACRO_LIB   = _libs[0] if _libs else ""
MACRO_TOPN  = "16"
print("macro library:", MACRO_LIB or "(NOT FOUND -> macro members degrade to plain seeds; floor unaffected)")

# v47: keep the proven v43 3-seed core (pop64/r10), replace the two macro members
# (no gain in v46) with two never-tried members: the thesis config (seed 47, pop 40,
# restarts 5) and the same with pixel fitness distance instead of graph distance.
# (seed, max_rules, macro_lib, pop, restarts, extra_args)
RUNS = [   # v71: v69's layout (v64-v69: 2.50; v70 without the v52 members: 1.67) with 47/41 on round58 ce4f16e (round57 + output-identical speed-ups, ~3x generations per second) + ARC_REENC_CACHE; v69: 777/2718/333 on the v52 code; 47/41 on 4f619f7 (round53: round52 DSL, 10x faster scene ops on big grids, Phase-A cache), 8-search ensemble + ARC_VOTE_TYPES + ARC_TASK_OTHER + ARC_MDL_PRIOR (ConceptARC e16 600 s 471/474 of 480; ARC-AGI-2 eval 2-4/120)
    (777,  5, "", "64",  "10", []),                                   # anchor -> attempt_1
    (47,   5, "", "40",  "5",  []),                                   # v63: pop 40, ensemble 8
    (41,   5, "", "40",  "5",  []),                                   # v63: pop 40, ensemble 8 (v68: before the v52 members, so it wins attempt_2 vote ties)
    (2718, 5, "", "640", "5",  []),                                   # NEW v52: pop 640
    (333,  5, "", "160", "5",  []),                                   # NEW v52: pop 160
]

with open(challenges_file) as f:
    challenges = json.load(f)
n_tasks = len(challenges)
rounds  = -(-n_tasks // N_WORKERS)
per_task = max(20, int(TOTAL_BUDGET_SECONDS / max(1, len(RUNS)) / max(1, rounds) * 0.95))
print(f"n_tasks={n_tasks}  N_WORKERS={N_WORKERS}  runs={len(RUNS)}  per_task_cap={per_task}s "
      f"(~{per_task*rounds/3600.0:.2f}h/run)")

# --- v48: skip the 11h search on the push/dev run; do it only in the scoring rerun ---
# Real submission if Kaggle flags a competition rerun OR the test file is not the known
# public 240-task sample. Fail-safe: any doubt -> full run (the old behaviour).
import hashlib
_PUBLIC_TEST_SHA1 = "f84bf592369136a67a6db2b76d9ee42fd2dca688"
_ids_sha1 = hashlib.sha1(",".join(sorted(challenges)).encode()).hexdigest()
IS_RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN")) or _ids_sha1 != _PUBLIC_TEST_SHA1
print(f"IS_RERUN={IS_RERUN}  env={os.environ.get('KAGGLE_IS_COMPETITION_RERUN')!r}  ids_sha1={_ids_sha1}")
DEV_ARGS = []
if not IS_RERUN:
    RUNS     = RUNS[:2]                 # smoke test: anchor + one round58 member (v71), 2 tasks, 20 s each
    per_task = 20
    rounds   = 1
    DEV_ARGS = ["--max-tasks", "2"]
    print("DEV run: smoke test only; the assemble cell fills placeholders for the rest")

out_path = os.path.join(WORK_DIR, "submission.json")
PLACEHOLDER = [[0, 0], [0, 0]]

def merge_and_write(subs):
    """attempt_1 = subs[0]'s primary (proven anchor). attempt_2 = the MOST-VOTED
    primary across the other runs that differs from attempt_1 (consensus filters
    config-specific overfits; macro solves get 2 votes from the two macro members),
    else subs[0]'s attempt_2, else attempt_1."""
    merged = {}
    for tid, task in challenges.items():
        entries = []
        for i in range(len(task["test"])):
            base = subs[0].get(tid) if subs else None
            a1 = base[i]["attempt_1"] if base and i < len(base) else PLACEHOLDER
            k1 = json.dumps(a1)
            votes, order = {}, {}
            for si, s in enumerate(subs[1:]):
                e = s.get(tid)
                if e and i < len(e):
                    cand = e[i]["attempt_1"]; k = json.dumps(cand)
                    if k == k1:
                        continue
                    votes[k] = votes.get(k, 0) + 1
                    if k not in order:
                        order[k] = (si, cand)
            if votes:
                a2 = order[min(votes, key=lambda k: (-votes[k], order[k][0]))][1]
            elif base and i < len(base) and json.dumps(base[i]["attempt_2"]) != k1:
                a2 = base[i]["attempt_2"]
            else:
                a2 = a1
            entries.append({"attempt_1": a1, "attempt_2": a2})
        merged[tid] = entries
    with open(out_path, "w") as f:
        json.dump(merged, f)
    return len(merged)

V52_SEEDS = {777, 2718, 333}   # v59: anchor members on the v52 code
seed_subs = []
for seed, mr, mlib, pop, rst, extra in RUNS:
    tag = f"s{seed}_mr{mr}_p{pop}_r{rst}" + ("_macro" if mlib else "") + ("_" + "".join(extra[1:]) if extra else "")
    sp  = os.path.join(WORK_DIR, f"sub_{tag}.json")
    env = dict(os.environ)
    v52_member = seed in V52_SEEDS
    if not v52_member:
        env["ARC_CONFIG"] = os.path.join(CSOLVER, "arc_config.conf")   # v55: same base config as the offline runs
        env["ARC_CONCEPT_SEEDS"] = "0"   # v57: concept ops only through evolution
        env["ARC_GENERALISE"] = "1"
        env["ARC_INFORMED_ATTR"] = "1"
        env["ARC_IMMIGRANTS"] = "4"   # v58
        env["ARC_NO_GUARDS"] = "1"   # v60: ConceptARC 352 -> 362
        env["ARC_ENSEMBLE"] = "8"   # v63: 8 searches on 1/8 of the budget, voted
        env["ARC_VOTE_TYPES"] = "1"   # v64: answers with a change type no train pair has count half
        env["ARC_TASK_OTHER"] = "1"   # v64: patch fill=other falls back to the train colours
        env["ARC_MDL_PRIOR"] = "1"   # v67: shorter programs win ties (ConceptARC 456 -> 471)
        env["ARC_REENC_CACHE"] = "1"   # v71: mid-program re-encodes through the scene cache (output-identical)
    if mlib:
        env["ARC_MACRO_SEED_LIBRARY"] = mlib
        env["ARC_MACRO_SEED_TOPN"]    = MACRO_TOPN
    cmd = [
        ARC_SOLVER_V52 if v52_member else ARC_SOLVER, "--kaggle", COMP_DIR, "--split", SPLIT,
        "--max-task-seconds", str(per_task), "--n-workers", str(N_WORKERS),
        "--pop", pop, "--generations", "10000000", "--restarts", rst,
        "--max-rules", str(mr), "--seed", str(seed),
        "--occam-continue", "1", "--view-continue", "1",
        "--out", sp,
    ] + extra + DEV_ARGS
    print(f"[{tag}] " + ("v52-code " if v52_member else "ensemble8 ") + (" macro=%s" % os.path.basename(mlib) if mlib else " macro=off"))
    print("      " + " ".join(cmd))
    t0 = time.time()
    try:
        rc = subprocess.run(cmd, cwd=WORK_DIR, env=env,
                            timeout=per_task * rounds + 600).returncode
        print(f"[{tag}] exit={rc} in {time.time() - t0:.0f}s")
    except subprocess.TimeoutExpired:
        print(f"[{tag}] timed out after {time.time() - t0:.0f}s")
    if os.path.isfile(sp):
        with open(sp) as f:
            seed_subs.append(json.load(f))
        n = merge_and_write(seed_subs)   # crash-safe: valid submission after each run
        print(f"  merged {len(seed_subs)} run(s) -> {out_path}: {n} tasks")


In [ ]:
# === Assemble + guarantee full coverage (+ score on the evaluation split) =====
# The C++ binary already writes a COMPLETE submission (every task placeholder-
# seeded; real attempts where it solved). This cell is a safety net: it re-checks
# coverage against the challenge file so a partial/failed run still yields a valid
# file, and scores against the solutions file when running the evaluation split.
submission = {}
if os.path.isfile(out_path):
    with open(out_path) as f:
        submission = json.load(f)
    print(f"Loaded solver submission.json: {len(submission)} tasks")
else:
    print("No submission.json from the solver -> filling all placeholders.")

with open(challenges_file) as f:
    challenges = json.load(f)

PLACEHOLDER = [[0, 0], [0, 0]]
filled = 0
for tid, task in challenges.items():
    n_test = len(task["test"])
    att = submission.get(tid)
    if not att or len(att) != n_test:
        submission[tid] = [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER}
                           for _ in range(n_test)]
        filled += 1
submission = {tid: submission[tid] for tid in challenges}   # exact key set + order

with open(out_path, "w") as f:
    json.dump(submission, f)
print(f"Wrote {out_path}: {len(submission)} tasks, {filled} placeholder-filled, "
      f"{len(submission) - filled} from the solver")

# Dry-run scoring: the evaluation split ships solutions (test does not).
sol_hits = _find(f"arc-agi_{SPLIT}_solutions.json")
if sol_hits:
    with open(sol_hits[0]) as f:
        solutions = json.load(f)
    solved = 0
    for tid, sols in solutions.items():
        preds = submission.get(tid, [])
        ok = all(
            i < len(preds) and sol in (preds[i]["attempt_1"], preds[i]["attempt_2"])
            for i, sol in enumerate(sols)
        )
        solved += int(ok)
    print(f"[{SPLIT}] solved {solved}/{len(solutions)} "
          f"({100.0 * solved / max(1, len(solutions)):.1f}%)")
